# Data Cleaning and Aggregation 


In [111]:
import sqlite3 
import pandas as pd

## Importing raw observations

In [112]:
conn = sqlite3.connect('../data/probes.db')
df_raw_observations = pd.read_sql_query("SELECT * FROM probes", conn)

## Cleaning

- During collection, the WiFi adapter failed at times, resulting in incomplete scan cycles.
- On average every scan cycle lasted 30-40 seconds (10 seconds for channel 1,6,11 + changing the channel latency), thus all cycles that are under 30 seconds are faulty cycles
- Since occupancy does not change drastically every 30 seconds, I decided to impute the faulty cycles with the previous/subsequent cycle (if the date matches)

In [113]:
df_scan = pd.read_sql_query("""
    SELECT 
        scan_cycle_id,
        timestamp,
        DATE(timestamp, 'unixepoch') AS date,
        ROUND(MAX(timestamp) - MIN(timestamp), 2) AS total_duration_seconds,
        COUNT(*) AS total_probes,
        COUNT(CASE WHEN rssi >= -55 THEN 1 END) AS probes_rssi_55, 
        COUNT(CASE WHEN rssi >= -60 THEN 1 END) AS probes_rssi_60, 
        COUNT(CASE WHEN rssi >= -65 THEN 1 END) AS probes_rssi_65, 
        COUNT(CASE WHEN rssi >= -70 THEN 1 END) AS probes_rssi_70, 
        COUNT(CASE WHEN rssi >= -75 THEN 1 END) AS probes_rssi_75,
        COUNT(CASE WHEN rssi >= -80 THEN 1 END) AS probes_rssi_80
    FROM probes 
    GROUP BY scan_cycle_id 
    ORDER BY scan_cycle_id
""", conn)

df_scan.head()

,scan_cycle_id,timestamp,date,total_duration_seconds,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,probes_rssi_80
0,1,1.787343e+09,2026-08-21,34.66,117,13,19,34,47,75,98
1,2,1.787343e+09,2026-08-21,35.78,110,5,19,31,54,79,99
2,3,1.787343e+09,2026-08-21,35.16,100,4,22,42,60,76,82
3,4,1.787343e+09,2026-08-21,35.58,67,5,7,18,28,53,63
4,5,1.787343e+09,2026-08-21,33.82,158,28,33,53,98,129,141


In [114]:
faulty_cycles = []
replacement = []
feature_columns = [col for col in df_scan if col not in ['scan_cycle_id', 'timestamp', 'date']]

for x in df_scan.index:
    prev_valid = (x > df_scan.index[0] and 
                     df_scan.loc[x, 'date'] == df_scan.loc[x-1, 'date'] and 
                     df_scan.loc[x-1, 'total_duration_seconds'] > 30)
    next_valid = (x < df_scan.index[-1] and 
              df_scan.loc[x, 'date'] == df_scan.loc[x+1, 'date'] and 
              df_scan.loc[x+1, 'total_duration_seconds'] > 30)
    if df_scan.loc[x, 'total_duration_seconds'] < 30:
        if prev_valid:
          faulty_cycles.append(x)
          replacement.append(x - 1)
          df_scan.loc[x, feature_columns] = df_scan.loc[x - 1, feature_columns]
        elif next_valid:
          faulty_cycles.append(x)
          replacement.append(x + 1)
          df_scan.loc[x, feature_columns] = df_scan.loc[x + 1, feature_columns]

print(f"Faulty cycles: {faulty_cycles}, Amount: {len(faulty_cycles)}")
df_scan.tail()

Faulty cycles: [103, 111, 179, 204, 366, 399, 513, 700, 705, 732, 844, 899, 910, 1019, 1065, 1074, 1113, 1449], Amount: 18


,scan_cycle_id,timestamp,date,total_duration_seconds,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,probes_rssi_80
1445,1446,1.788459e+09,2026-09-03,37.87,324,22,52,126,183,232,253
1446,1447,1.788459e+09,2026-09-03,37.93,385,27,29,41,104,209,292
1447,1448,1.788459e+09,2026-09-03,38.12,304,7,23,79,190,261,288
1448,1449,1.788459e+09,2026-09-03,39.42,320,4,12,48,137,253,298
1449,1450,1.788459e+09,2026-09-03,39.42,320,4,12,48,137,253,298


## Feature Aggregation

- Occupancy was recorded every 5 minutes during data collection, 
  and the same interval is used as the base unit for the prediction model.

- The 1,450 raw scan cycles (30 seconds each) were aggregated into 
  5-minute windows of 10 cycles each, producing ~145 labelled observations.

- Probe-based features (filtered counts, RSSI thresholds) were computed 
  by summing across all valid cycles within each window.

- MAC address metrics were computed directly from `df_raw_observations` 
  by counting distinct addresses within each 5-minute window, avoiding 
  the risk of duplicate counting that would arise from cycle-level aggregation.

- 18 faulty cycles (<2% of data) were identified by duration below 30 
  seconds. Probe-based features for faulty cycles were imputed from the 
  nearest valid cycle on the same day. MAC address metrics were not imputed 
  since faulty cycles still captured real devices — excluding them would 
  undercount rather than improve accuracy.

- Room capacity was added as a feature to account for the multi-environment 
  nature of the dataset. The target variable was normalised to occupancy 
  ratio (ground_truth_people / room_capacity) to enable generalisation 
  across spaces of different sizes.

In [ ]:
mac_addresses = pd.read_sql_query(
    """
    WITH mac_counts AS (
    SELECT 
        (scan_cycle_id) / 10 AS window_id,
        mac_address,
        COUNT(*) AS times_seen
    FROM probes
    GROUP BY window_id, mac_address
    )
    SELECT 
        window_id,
        COUNT(mac_address) AS unique_mac_addresses,
        COUNT(CASE WHEN times_seen >= 2 THEN 1 END) AS unique_mac_2plus,
        COUNT(CASE WHEN times_seen >= 3 THEN 1 END) AS unique_mac_3plus,
        COUNT(CASE WHEN times_seen >= 4 THEN 1 END) AS unique_mac_4plus,
        COUNT(CASE WHEN times_seen >= 5 THEN 1 END) AS unique_mac_5plus
    FROM mac_counts
    GROUP BY window_id
    ORDER BY window_id
    """,
    conn,
)

conn.close()

df_5min = df_scan.groupby((df_scan["scan_cycle_id"] - 1) // 10 + 1).agg(
    timestamp=("timestamp", "first"),
    total_probes=("total_probes", "sum"),
    probes_rssi_55=("probes_rssi_55", "sum"),
    probes_rssi_60=("probes_rssi_60", "sum"),
    probes_rssi_65=("probes_rssi_65", "sum"),
    probes_rssi_70=("probes_rssi_70", "sum"),
    probes_rssi_75=("probes_rssi_75", "sum"),
    probes_rssi_80=("probes_rssi_80", "sum")
)
mac_addresses.set_index("window_id", inplace=True)
df = pd.merge(df_5min, mac_addresses, left_index=True, right_index=True)

df["max_capacity"] = 0

df.loc[1:18, "max_capacity"] = 65
df.loc[19:27, "max_capacity"] = 40
df.loc[28:42, "max_capacity"] = 50
df.loc[43:70, "max_capacity"] = 70
df.loc[71:72, "max_capacity"] = 45
df.loc[73:88, "max_capacity"] = 64
df.loc[89:113, "max_capacity"] = 50
df.loc[114:132, "max_capacity"] = 300
df.loc[133:145, "max_capacity"] = 600

df["ground_truth_people"] = [22,29,31,26,28,27,37,25,25,37,28,27,25,26,33,45,34,28,6,6,4,5,6,4,5,7,10,12,13,15,16,17,17,14,16,8,5,7,6,7,5,6,19,22,20,22,25,28,29,29,32,34,35,38,43,48,50,42,42,43,48,49,52,52,52,52,43,45,54,52,8,8,10,12,12,12,9,10,8,8,8,8,10,11,11,12,10,6,7,6,7,8,8,6,7,6,6,8,8,11,9,9,8,8,10,10,10,6,4,4,7,6,8,112,119,106,105,125,108,113,111,90,86,72,61,46,45,47,46,50,48,39,59,74,101,124,144,165,162,184,251,272,281,302,318]
df["occupancy_ratio"] = df["ground_truth_people"] / df["max_capacity"]

df.head()

,timestamp,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,probes_rssi_80,unique_mac_addresses,unique_mac_2plus,unique_mac_3plus,unique_mac_4plus,unique_mac_5plus,max_capacity,ground_truth_people,occupancy_ratio
scan_cycle_id,,,,,,,,,,,,,,,,
1,1.787343e+09,1192,95,192,380,615,863,1032,608,168,51,26,24,65,22,0.338462
2,1.787343e+09,1083,42,143,297,554,852,973,653,187,45,26,15,65,29,0.446154
3,1.787344e+09,1200,133,234,404,643,925,1083,523,155,49,31,18,65,31,0.476923
4,1.787344e+09,924,23,70,218,456,649,790,470,120,36,27,21,65,26,0.400000
5,1.787345e+09,966,50,107,204,413,701,868,600,174,43,23,17,65,28,0.430769


## Anomaly filtering

- During EDA, scan cycle 145 was identified as a sensor anomaly and removed prior to model training. 
- The cycle exhibited an abnormally high probe-to-unique-MAC ratio of 73.29 
- Approximately 12 standard deviations above the dataset mean of 3.04 ± 5.95
- This indicates a malfunctioning or rogue device rather than genuine human presence.

In [116]:
ratio = df['total_probes'] / df['unique_mac_addresses']
ratio.describe()

count    145.000000
mean       3.044079
std        5.949894
min        0.565217
25%        1.935780
50%        2.354839
75%        3.040956
max       73.291667
dtype: float64

In [117]:
ratio = ratio.sort_values(ascending=False)
ratio.head()

scan_cycle_id
145    73.291667
40      5.990991
70      5.957547
41      5.705882
27      5.646766
dtype: float64

In [118]:
df.drop(index=145, inplace=True)
df.to_pickle("../data/processed/df.pkl")
df.tail()

,timestamp,total_probes,probes_rssi_55,probes_rssi_60,probes_rssi_65,probes_rssi_70,probes_rssi_75,probes_rssi_80,unique_mac_addresses,unique_mac_2plus,unique_mac_3plus,unique_mac_4plus,unique_mac_5plus,max_capacity,ground_truth_people,occupancy_ratio
scan_cycle_id,,,,,,,,,,,,,,,,
140,1.788456e+09,2822,25,177,650,1299,1898,2291,1540,463,163,94,60,600,184,0.306667
141,1.788457e+09,3000,27,196,682,1356,2036,2551,1734,522,190,107,67,600,251,0.418333
142,1.788457e+09,3308,60,329,949,1810,2525,3005,1553,560,226,147,102,600,272,0.453333
143,1.788458e+09,3326,65,274,862,1673,2340,2835,1768,569,240,134,92,600,281,0.468333
144,1.788458e+09,3678,55,281,991,1963,2826,3339,1834,587,229,135,87,600,302,0.503333
